# 02 - SFT 数据混合实验

## 核心问题：为什么数据混合比例如此重要？

Tülu 3 论文 Section 3 揭示了一个关键发现：

> **"Skill Isolation"** — 不同类型的数据（数学、代码、安全、对话）各自提升对应能力，
> 而安全数据几乎不影响通用能力（Safety Orthogonality）。

本 Notebook 通过三组消融实验验证这一发现：
- **实验 A**：仅用安全数据（wildguardmix + wildjailbreak + coconot）
- **实验 B**：仅用能力数据（flan_v2 + numinamath + no_robots）
- **实验 C**：完整混合数据（全部 10 个子集）

通过对比三组的 loss 曲线和生成质量，验证 Skill Isolation 假设。

In [ ]:
import json
import os
import sys
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['Arial Unicode MS', 'sans-serif']

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## A. 数据组成分析

先看我们的 SFT 数据由哪些子集构成，以及每类数据的作用。

In [ ]:
# 加载 SFT 数据
sft_path = PROJECT_ROOT / "data/sft_mix/train.jsonl"
samples = []
with open(sft_path, "r") as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

print(f"SFT 总样本数: {len(samples):,}")

# 按 source 统计
source_counts = Counter(s.get("source", "unknown") for s in samples)

# 定义技能分类
SKILL_CATEGORIES = {
    "安全类": ["wildguardmix", "wildjailbreak", "coconot"],
    "能力类": ["flan_v2", "numinamath", "no_robots", "openassistant"],
    "合成类": ["persona_if", "persona_math", "persona_code"],
}

# 显示分类统计
print("\n" + "="*50)
print("按技能分类统计:")
print("="*50)
for category, sources in SKILL_CATEGORIES.items():
    total = sum(source_counts.get(s, 0) for s in sources)
    print(f"\n{category} ({total} 条):")
    for s in sources:
        count = source_counts.get(s, 0)
        print(f"  {s}: {count}")

In [ ]:
# 可视化：数据混合比例饼图 + 技能分类
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 左图：按子集的饼图
sorted_sources = sorted(source_counts.items(), key=lambda x: -x[1])
labels = [s[0] for s in sorted_sources]
sizes = [s[1] for s in sorted_sources]
colors = plt.cm.Set3(range(len(labels)))

axes[0].pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('SFT Data Mix: By Subset', fontsize=14, fontweight='bold')

# 右图：按技能分类的柱状图
cat_names = []
cat_totals = []
cat_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
for category, sources in SKILL_CATEGORIES.items():
    cat_names.append(category)
    cat_totals.append(sum(source_counts.get(s, 0) for s in sources))

bars = axes[1].bar(cat_names, cat_totals, color=cat_colors, edgecolor='black', linewidth=0.5)
for bar, total in zip(bars, cat_totals):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
                f'{total}', ha='center', va='bottom', fontweight='bold')
axes[1].set_title('SFT Data Mix: By Skill Category', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Sample Count')

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'results/figures/sft_data_mix.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to results/figures/sft_data_mix.png")

## B. 三组消融数据准备

将数据拆分为三组，模拟 Tülu 3 的 Skill Isolation 实验：

| 实验组 | 数据来源 | 验证目标 |
|--------|----------|----------|
| Safety-Only | wildguardmix + wildjailbreak + coconot | 安全拒绝能力 |
| Capability-Only | flan_v2 + numinamath + no_robots + openassistant | 通用 + 数学 + 对话 |
| Full Mix | 全部 10 个子集 | 完整能力 |

In [ ]:
# 拆分三组数据
safety_sources = {"wildguardmix", "wildjailbreak", "coconot"}
capability_sources = {"flan_v2", "numinamath", "no_robots", "openassistant"}

safety_only = [s for s in samples if s.get("source") in safety_sources]
capability_only = [s for s in samples if s.get("source") in capability_sources]
full_mix = samples  # 全部

print(f"Safety-Only:     {len(safety_only):,} 条")
print(f"Capability-Only: {len(capability_only):,} 条")
print(f"Full Mix:        {len(full_mix):,} 条")

# 每组采样一些看看内容
import random
random.seed(42)

print("\n" + "="*60)
print("Safety-Only 样本示例 (前2条):")
print("="*60)
for s in random.sample(safety_only, min(2, len(safety_only))):
    msgs = s["messages"]
    user_msg = next((m["content"] for m in msgs if m["role"] == "user"), "N/A")
    asst_msg = next((m["content"] for m in msgs if m["role"] == "assistant"), "N/A")
    print(f"  Source: {s.get('source')}")
    print(f"  User: {user_msg[:120]}...")
    print(f"  Asst: {asst_msg[:120]}...")
    print()

print("="*60)
print("Capability-Only 样本示例 (前2条):")
print("="*60)
for s in random.sample(capability_only, min(2, len(capability_only))):
    msgs = s["messages"]
    user_msg = next((m["content"] for m in msgs if m["role"] == "user"), "N/A")
    asst_msg = next((m["content"] for m in msgs if m["role"] == "assistant"), "N/A")
    print(f"  Source: {s.get('source')}")
    print(f"  User: {user_msg[:120]}...")
    print(f"  Asst: {asst_msg[:120]}...")
    print()

## C. Token 长度分布对比

不同类型数据的 token 长度分布差异很大，这会影响训练效率。

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B", trust_remote_code=True)

def get_token_lengths(data, max_samples=500):
    """计算每条样本的 token 数"""
    lengths = []
    for s in data[:max_samples]:
        text = ""
        for msg in s["messages"]:
            text += msg.get("content", "") + " "
        tokens = tokenizer.encode(text, add_special_tokens=False)
        lengths.append(len(tokens))
    return lengths

safety_lens = get_token_lengths(safety_only)
cap_lens = get_token_lengths(capability_only)
full_lens = get_token_lengths(full_mix)

# 统计
import numpy as np
for name, lens in [("Safety-Only", safety_lens), ("Capability-Only", cap_lens), ("Full Mix", full_lens)]:
    arr = np.array(lens)
    print(f"{name:20s}: mean={arr.mean():.0f}, median={np.median(arr):.0f}, "
          f"p90={np.percentile(arr, 90):.0f}, max={arr.max():.0f}")

In [ ]:
# Token 长度分布直方图
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(safety_lens, bins=50, alpha=0.6, label=f'Safety-Only (n={len(safety_lens)})', color='#FF6B6B')
ax.hist(cap_lens, bins=50, alpha=0.6, label=f'Capability-Only (n={len(cap_lens)})', color='#4ECDC4')
ax.hist(full_lens, bins=50, alpha=0.4, label=f'Full Mix (n={len(full_lens)})', color='#45B7D1')

ax.axvline(x=1024, color='red', linestyle='--', label='max_seq_length=1024')
ax.set_xlabel('Token Count', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Token Length Distribution: Safety vs Capability vs Full Mix', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'results/figures/token_length_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to results/figures/token_length_distribution.png")

## D. Skill Isolation 验证思路

### Tülu 3 的核心发现

论文中的关键实验结果：

```
+------------------+----------+----------+----------+
| 训练数据          | HellaSwag | Safety   | Math     |
+------------------+----------+----------+----------+
| Safety-Only      | ~不变     | ↑↑↑      | ~不变     |
| Capability-Only  | ↑↑       | ~不变     | ↑↑       |
| Full Mix         | ↑↑       | ↑↑↑      | ↑↑       |
+------------------+----------+----------+----------+
```

### Safety Orthogonality（安全正交性）

这是 Tülu 3 最重要的发现之一：

- **加入安全数据不会降低通用能力** → 可以放心加安全数据
- **安全数据只提升安全相关指标** → 技能是隔离的
- **这使得 "先能力后安全" 的两阶段训练没有必要** → 可以一次混合训练

### 在我们的 Smoke Test 中验证

虽然 smoke_test 数据量小（2000条），但我们仍可以通过以下方式验证：

1. **Loss 对比**：不同数据组的训练 loss 下降速度
2. **生成对比**：用相同 prompt 对比三组模型的输出
3. **拒绝率对比**：用有害 prompt 测试拒绝率

In [ ]:
# 为每个子集计算平均 token 长度
source_avg_lengths = {}
for source in source_counts.keys():
    subset = [s for s in samples if s.get("source") == source]
    lens = get_token_lengths(subset, max_samples=200)
    if lens:
        source_avg_lengths[source] = np.mean(lens)

# 按平均 token 长度排序的柱状图
fig, ax = plt.subplots(figsize=(14, 6))

sorted_items = sorted(source_avg_lengths.items(), key=lambda x: -x[1])
names = [item[0] for item in sorted_items]
avg_lens = [item[1] for item in sorted_items]

# 按技能分类着色
bar_colors = []
for name in names:
    if name in safety_sources:
        bar_colors.append('#FF6B6B')
    elif name in capability_sources:
        bar_colors.append('#4ECDC4')
    else:
        bar_colors.append('#45B7D1')

bars = ax.bar(names, avg_lens, color=bar_colors, edgecolor='black', linewidth=0.5)
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_ylabel('Average Token Length', fontsize=12)
ax.set_title('Average Token Length by Subset', fontsize=14, fontweight='bold')

# 图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#FF6B6B', label='Safety'),
    Patch(facecolor='#4ECDC4', label='Capability'),
    Patch(facecolor='#45B7D1', label='Synthetic'),
]
ax.legend(handles=legend_elements, fontsize=10)

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'results/figures/avg_token_by_subset.png'), dpi=150, bbox_inches='tight')
plt.show()

## E. 关键要点总结

### 从数据分析中学到的

1. **安全类数据** 通常较短（wildjailbreak/wildguardmix 多为单轮对话），training 速度快
2. **能力类数据** 长度差异大（flan_v2 短、numinamath 含详细推理过程较长）
3. **合成数据** (persona_*) 是 Tülu 3 的创新——用 LLM 合成高质量训练数据

### 对混合策略的影响

- 数据混合不是简单拼接，需要考虑各子集的 **规模比例** 和 **token 长度分布**
- Tülu 3 的策略：先各技能独立调参，找到最优配比，再合并训练
- 我们的 smoke_test 使用论文推荐的近似比例，在 Notebook 07 中将做完整消融实验

### 下一步

→ **Notebook 03**：使用完整混合数据进行 SFT 训练，观察 loss 曲线和 Base vs SFT 对比